In [1]:
# ============================================
# DFD03 Celeb-DF v2 — PREPROCESSING
# ============================================
# Input: "celeb-df-v2" dataset from Kaggle
# Output: preprocessed_CelebDF_16.tar.gz + preprocessed_CelebDF_32.tar.gz
#
# Celeb-DF v2 has folders: Celeb-real/, Celeb-synthesis/, YouTube-real/
# Our preprocess_videos.py expects: real/ and fake/
# So we create symlinks to map the structure.
#
# We sample ~3300 videos (balanced with DFDC02/DFD01 sizes)
# from the full 6229 to keep datasets equal.
#
# WHY limit fake to 2710 instead of all 5639:
# 1. Balance: 590 real vs 5639 fake = 1:9.6 ratio → model overfits on real
#    (sees same 590 real ~10x per epoch). With 2710 fake → 1:4.6, manageable.
# 2. Dataset parity: DFDC02=3240, DFD01=3430, CelebDF=3300 — equal weight
#    in 3-dataset training, no single dataset dominates.
# 3. Disk: 6229 × 32 frames ≈ 40GB → exceeds Kaggle 20GB working space.
#
# IMPORTANT: Pillow must be downgraded BEFORE other imports
# because Kaggle's pre-installed Pillow 11+ breaks torchvision.

NUM_FRAMES_LIST = [16, 32]  # Process both T=16 and T=32
OUTPUT_SIZE = 224
MIN_FACE_CONF = 0.90
MIN_DETECTION_RATIO = 0.55
DETECTOR_MAX_SIDE = 960

# Limit videos to balance with DFDC02 (~3300) and DFD01 (~3400)
MAX_REAL = 590       # Take ALL real (only 590 available)
MAX_FAKE = 2710      # Sample from 5639 → total ~3300

import subprocess, sys, os, time, shutil, json, random
import glob as glob_mod

random.seed(42)  # Reproducible sampling

# Step 1: Force downgrade Pillow FIRST
print('=== Downgrading Pillow ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '--force-reinstall', 'Pillow==10.4.0'],
               check=True)
print('Pillow downgraded.')

# Step 2: Install other dependencies
print('=== Installing dependencies ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy<2', 'scipy<1.17', 'scikit-learn', 'timm', 'facenet-pytorch',
                'opencv-python-headless', 'tqdm'],
               check=True)
print('Dependencies installed.')

# Step 3: Clone project
if not os.path.exists('/kaggle/working/project/.git'):
    subprocess.run(['rm', '-rf', '/kaggle/working/project'], check=False)
    subprocess.run(['git', 'clone', 'https://github.com/Jokeera/deepfake-detection.git',
                     '/kaggle/working/project'], check=True)
    print('Project cloned.')
else:
    subprocess.run(['git', '-C', '/kaggle/working/project', 'pull', '--ff-only'],
                   check=True)
    print('Project updated.')

# Step 4: Explore Celeb-DF v2 dataset structure
print('\n=== Exploring Celeb-DF v2 dataset structure ===')
celeb_input = None
for candidate in [
    '/kaggle/input/celeb-df-v2',
    '/kaggle/input/datasets/reubensuju/celeb-df-v2',
]:
    if os.path.exists(candidate):
        celeb_input = candidate
        print(f'Found dataset at: {celeb_input}')
        break

if celeb_input is None:
    print('Searching for dataset...')
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'celeb' in root.lower():
            celeb_input = root
            print(f'Found: {celeb_input}')
            break

if celeb_input is None:
    raise RuntimeError('Celeb-DF v2 dataset not found!')

# Show structure
print('\nDataset structure:')
VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv', '.webm', '.m4v'}
for root, dirs, files in os.walk(celeb_input):
    level = root.replace(celeb_input, '').count(os.sep)
    if level < 3:
        vids = sum(1 for f in files if os.path.splitext(f)[1].lower() in VIDEO_EXTS)
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/ ({len(dirs)} dirs, {vids} videos, {len(files)} files)')

# Step 5: Collect all video paths by label
real_videos = []
fake_videos = []

for root, dirs, files in os.walk(celeb_input):
    dirname = os.path.basename(root).lower()
    
    for f in files:
        if os.path.splitext(f)[1].lower() not in VIDEO_EXTS:
            continue
        
        src = os.path.join(root, f)
        
        if 'real' in dirname or 'youtube' in dirname or 'original' in dirname:
            real_videos.append((src, f, dirname))
        elif 'synthesis' in dirname or 'fake' in dirname or 'swap' in dirname or 'manipulated' in dirname:
            fake_videos.append((src, f, dirname))
        else:
            print(f'  SKIP unknown folder: {root}/{f}')

print(f'\nFound: {len(real_videos)} real, {len(fake_videos)} fake')

# Step 6: Sample to balance with other datasets
if len(real_videos) > MAX_REAL:
    real_videos = random.sample(real_videos, MAX_REAL)
if len(fake_videos) > MAX_FAKE:
    fake_videos = random.sample(fake_videos, MAX_FAKE)

print(f'Sampled: {len(real_videos)} real, {len(fake_videos)} fake = {len(real_videos) + len(fake_videos)} total')
print(f'(Balanced with DFDC02 ~3300, DFD01 ~3400)')

# Step 7: Create symlinked structure real/ + fake/
PREPARED_DIR = '/kaggle/working/celeb_prepared'
if os.path.exists(PREPARED_DIR):
    shutil.rmtree(PREPARED_DIR)
os.makedirs(os.path.join(PREPARED_DIR, 'real'), exist_ok=True)
os.makedirs(os.path.join(PREPARED_DIR, 'fake'), exist_ok=True)

for src, f, dirname in real_videos:
    dst = os.path.join(PREPARED_DIR, 'real', f)
    if os.path.exists(dst):
        name, ext = os.path.splitext(f)
        dst = os.path.join(PREPARED_DIR, 'real', f'{name}_{dirname}{ext}')
    os.symlink(src, dst)

for src, f, dirname in fake_videos:
    dst = os.path.join(PREPARED_DIR, 'fake', f)
    if os.path.exists(dst):
        name, ext = os.path.splitext(f)
        dst = os.path.join(PREPARED_DIR, 'fake', f'{name}_{dirname}{ext}')
    os.symlink(src, dst)

print(f'Symlinks created in {PREPARED_DIR}')

# Step 8: Process for each T value
for NUM_FRAMES in NUM_FRAMES_LIST:
    output_name = f'preprocessed_CelebDF_{NUM_FRAMES}'
    output_dir = f'/kaggle/working/{output_name}'
    
    print(f'\n{"="*60}')
    print(f'PREPROCESSING Celeb-DF v2 T={NUM_FRAMES}')
    print(f'  Input: {PREPARED_DIR}')
    print(f'  Output: {output_dir}')
    print(f'  Videos: {len(real_videos)} real + {len(fake_videos)} fake = {len(real_videos) + len(fake_videos)}')
    print(f'{"="*60}\n')
    
    cmd = [
        sys.executable, '-u',
        '/kaggle/working/project/preprocess_videos.py',
        PREPARED_DIR,
        output_dir,
        '--max-frames', str(NUM_FRAMES),
        '--output-size', str(OUTPUT_SIZE),
        '--min-face-confidence', str(MIN_FACE_CONF),
        '--min-detection-ratio', str(MIN_DETECTION_RATIO),
        '--detector-max-side', str(DETECTOR_MAX_SIDE),
        '--device', 'cuda',
    ]
    
    print(f'Command: {" ".join(cmd)}')
    
    t0 = time.time()
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
        universal_newlines=True,
    )
    for line in proc.stdout:
        line = line.rstrip('\n')
        if line:
            print(line, flush=True)
    proc.wait()
    elapsed = (time.time() - t0) / 60
    
    if proc.returncode != 0:
        print(f'\nERROR: preprocessing T={NUM_FRAMES} failed (code {proc.returncode})')
        print('Continuing to next T value...')
        continue
    
    print(f'\nT={NUM_FRAMES} done in {elapsed:.1f} min')
    
    # Count results
    real_out = len([d for d in os.listdir(os.path.join(output_dir, 'real'))
                    if os.path.isdir(os.path.join(output_dir, 'real', d))]) if os.path.isdir(os.path.join(output_dir, 'real')) else 0
    fake_out = len([d for d in os.listdir(os.path.join(output_dir, 'fake'))
                    if os.path.isdir(os.path.join(output_dir, 'fake', d))]) if os.path.isdir(os.path.join(output_dir, 'fake')) else 0
    print(f'  Real: {real_out}, Fake: {fake_out}, Total: {real_out + fake_out}')
    
    # Package
    tar_name = f'{output_name}.tar.gz'
    tar_path = f'/kaggle/working/{tar_name}'
    print(f'\nPackaging {tar_name}...')
    os.system(f'tar czf {tar_path} -C /kaggle/working {output_name}/')
    tar_size = os.path.getsize(tar_path) / (1024**2)
    print(f'{tar_name}: {tar_size:.1f} MB')

# Final summary
print(f'\n{"="*60}')
print('CELEB-DF V2 PREPROCESSING COMPLETE')
print(f'{"="*60}')
for NUM_FRAMES in NUM_FRAMES_LIST:
    tar_name = f'preprocessed_CelebDF_{NUM_FRAMES}.tar.gz'
    tar_path = f'/kaggle/working/{tar_name}'
    if os.path.exists(tar_path):
        sz = os.path.getsize(tar_path) / (1024**2)
        print(f'  {tar_name}: {sz:.1f} MB')
print('Upload these as Kaggle datasets for multi-dataset training.')

=== Downgrading Pillow ===
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 61.5 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


Pillow downgraded.
=== Installing dependencies ===
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 4.1 MB/s eta 0:00:0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you 

Dependencies installed.


Cloning into '/kaggle/working/project'...


Project cloned.

=== Exploring Celeb-DF v2 dataset structure ===
Found dataset at: /kaggle/input/datasets/reubensuju/celeb-df-v2

Dataset structure:
celeb-df-v2/ (3 dirs, 0 videos, 1 files)
  YouTube-real/ (0 dirs, 300 videos, 300 files)
  Celeb-synthesis/ (0 dirs, 5639 videos, 5639 files)
  Celeb-real/ (0 dirs, 590 videos, 590 files)

Found: 890 real, 5639 fake
Sampled: 590 real, 2710 fake = 3300 total
(Balanced with DFDC02 ~3300, DFD01 ~3400)
Symlinks created in /kaggle/working/celeb_prepared

PREPROCESSING Celeb-DF v2 T=16
  Input: /kaggle/working/celeb_prepared
  Output: /kaggle/working/preprocessed_CelebDF_16
  Videos: 590 real + 2710 fake = 3300

Command: /usr/bin/python3 -u /kaggle/working/project/preprocess_videos.py /kaggle/working/celeb_prepared /kaggle/working/preprocessed_CelebDF_16 --max-frames 16 --output-size 224 --min-face-confidence 0.9 --min-detection-ratio 0.55 --detector-max-side 960 --device cuda
[INFO] Device: cuda
[INFO] Найдено видео: 3300
Preprocessing videos: 